# Dataset description

In [ ]:
# Choose dataset to analyze
DATASET_NAME = "../data/2_eurostat_new_passenger_cars_by_type_of_motor_energy.csv"

In [ ]:
import pandas as pd

df = pd.read_csv(DATASET_NAME)

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
for col in df.columns:
    print(f"Column {col}: {df[col].unique()}")

In [ ]:
df["TIME_PERIOD"].min(), df["TIME_PERIOD"].max()

In [ ]:
df["geo"].nunique(), df["Geopolitical entity (reporting)"].unique()

### Map 'geo' countries to continent

In [ ]:
import country_converter as coco

df_continent = pd.DataFrame({
    "geo": df["geo"].drop_duplicates()
})

df_continent = df_continent[df_continent["geo"] != "EU27_2020"]

df_continent["continent"] = coco.convert(
    names=df_continent["geo"],
    to="continent",
    not_found="Unknown"
)

# Count the number of countries per continent
df_continent.groupby("continent").size().reset_index(name="count")

# Data inspection, cleaning and preprocessing
## Drop unnecessary columns

In [ ]:
# Drop columns with a single unique value
df.drop(columns=[
    'STRUCTURE',
    'STRUCTURE_ID',
    'STRUCTURE_NAME',
    'freq',
    'Time frequency',
    'unit',
    'Unit of measure',
    'Time',
    'Observation value',
    'CONF_STATUS',
    'Confidentiality status (flag)'
], inplace=True)

# Drop description columns
df.drop(columns=[
    'mot_nrg',
    'Observation status (Flag) V2 structure'
], inplace=True)

df.info()

In [ ]:
df['OBS_VALUE'].min(), df['OBS_VALUE'].max()

In [ ]:
df['TIME_PERIOD'].sort_values().unique()

In [ ]:
for col in df.columns:
    print(f"Column {col}: {df[col].unique()}")

### Zeros observations

In [ ]:
import matplotlib.pyplot as plt

zeros = df[df["OBS_VALUE"] == 0]
zero_counts = (
    zeros.groupby(["Geopolitical entity (reporting)", "TIME_PERIOD"])
    .size()
    .reset_index(name="zero_count")
)
zero_counts["Geopolitical entity (reporting)"] = zero_counts["Geopolitical entity (reporting)"].replace("Kosovo*", "Kosovo")

plt.figure(figsize=(7, 8))

plt.scatter(
    zero_counts["TIME_PERIOD"],
    zero_counts["Geopolitical entity (reporting)"],
    s=zero_counts["zero_count"] * 10,  # bubble size * 10 to improve visibility
    color="blue"
)

plt.xlabel("Year")
plt.ylabel("Country")
plt.title("Distribution of Zeros in OBS_VALUE")

plt.tight_layout()
plt.savefig("img/zeros_distrib_in_OBS_VALUE.png", dpi=300)
plt.show()

In [ ]:
import matplotlib.pyplot as plt

total_registrations_per_country = (
    df[df["Motor energy"] == "Total"]
    .groupby("Geopolitical entity (reporting)")["OBS_VALUE"]
    .sum()
)
zero_observations_per_country = (
    df[df["OBS_VALUE"] == 0]
    .groupby("Geopolitical entity (reporting)")
    .size()
)

# [total_registrations, zero_observations]
plot_df = pd.DataFrame({
    "total_registrations": total_registrations_per_country,
    "zero_observations": zero_observations_per_country
}).fillna(0)
plot_df.rename(
    index={"Kosovo*": "Kosovo"},
    inplace=True
)
plot_df = plot_df.drop(
    index="European Union - 27 countries (from 2020)",
    errors="ignore"
)
plot_df = plot_df.sort_values("total_registrations")


fig, ax = plt.subplots(figsize=(7, 8))

# Blue bars for countries with zero observations, light gray for others
bar_colors = [
    "blue" if z > 0 else "lightgray"
    for z in plot_df["zero_observations"]
]

ax.barh(
    plot_df.index,
    plot_df["total_registrations"] / 1e6,
    color=bar_colors
)

# Labels for countries with zero observations
for country, registrations, zeros in zip(
    plot_df.index,
    plot_df["total_registrations"] / 1e6,
    plot_df["zero_observations"]
):
    if zeros > 0:
        ax.text(
            registrations,
            country,
            f"  {int(zeros)}",
            va="center",
            fontsize=9,
            fontweight="bold",
            color="blue"
        )

ax.set_xlabel("Total Car Registrations ($\\times10^6$)")
ax.set_ylabel("Country")

ax.set_title("Total Car Registrations and Zeros by Country")

plt.tight_layout()
plt.savefig("img/total_car_registrations_and_zeros_by_country.png", dpi=300)
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# [total_observations, zero_observations]
plot_df = (
    df.groupby("Motor energy")["OBS_VALUE"]
    .agg(
        total_observations="count",
        zero_observations=lambda x: (x == 0).sum()
    )
)

# [total_observations, zero_observations, zero_percentage]
plot_df["zero_percentage"] = (
    plot_df["zero_observations"] /
    plot_df["total_observations"] * 100
)

plot_df.sort_values("zero_percentage", inplace=True)

plt.figure(figsize=(7, 5))

plt.barh(
    plot_df.index,
    plot_df["zero_percentage"],
    color="lightgrey"
)

# Add percentage labels
for category, value in zip(plot_df.index, plot_df["zero_percentage"]):
    # To avoid overlapping with the plot borders
    # If value is > 85, put it inside the bar
    if value > 85:
        plt.text(
            value - 0.5,
            category,
            f"{value:.1f}$\%$",
            va="center",
            ha="right",
            fontsize=9,
            fontweight="bold",
            color="black"
        )
    # Otherwise put it outside the bar
    else:
        plt.text(
            value,
            category,
            f" {value:.1f}$\%$",
            va="center",
            ha="left",
            fontsize=9,
            fontweight="bold"
        )

plt.xlim(0, 100)
plt.xticks(range(0, 101, 10))

plt.xlabel("Zeros ($\%$)")
plt.ylabel("Motor energy category")
plt.title("Percentage of Zeros by motor energy category")

plt.tight_layout()
plt.savefig("img/percentage_of_zeros_by_motor_energy_category.png", dpi=300)
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Mean registrations per year and motor energy category
pivot_df = df.groupby(["TIME_PERIOD", "Motor energy"])["OBS_VALUE"].mean().unstack()

pivot_df.plot(figsize=(12, 6))

plt.xlabel("Year")
plt.ylabel("Mean number of registrations")
plt.title("Evolution of mean passenger car registrations by motor energy category")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

### NaN observations for the "OBS_VALUE" column

In [ ]:
# Check for NaN values in the OBS_VALUE column
nan_obs_value = df[df["OBS_VALUE"].isna()]

display(nan_obs_value)

In [ ]:
# Check Bi-fuel, Norway for all years
df[(df["Motor energy"] == "Bi-fuel") & (df["geo"] == "NO")]

In [ ]:
# Substitute NaN for <Bi-fuel, NO, 2023> with 0.0
df.loc[(df["Motor energy"] == "Bi-fuel") & (df["geo"] == "NO") & (df["TIME_PERIOD"] == 2023), "OBS_VALUE"] = 0.0
df[(df["Motor energy"] == "Bi-fuel") & (df["geo"] == "NO")]

### Flags

- (i) is value imputed by Eurostat or other receiving agencies
- (e) is estimated
- (b) is break in time series
- (m) is missing value; data cannot exist.

In [ ]:
flag_counts = (
    df["OBS_FLAG"]
    .value_counts(dropna=False)
    .rename_axis("Flag")
    .reset_index(name="Count")
)

flag_counts["Percentage"] = (
    flag_counts["Count"] / len(df) * 100
)

display(flag_counts)

In [ ]:
df[df["OBS_FLAG"] == "b"].sort_values(
    ["geo", "TIME_PERIOD"]
)

In [ ]:
df[df["OBS_FLAG"]=="b"].groupby("TIME_PERIOD").size() # As most of the "b" flags are in 2013 it is suggested to consider data from 2014 onwards for analysis

### Drop flag column

In [ ]:
df.drop(columns=[
    'OBS_FLAG'
], inplace=True)

### Restrict from 2014 to 2024

In [ ]:
df = df[df["TIME_PERIOD"] >= 2014]
df = df[df["TIME_PERIOD"] <= 2024]

df["TIME_PERIOD"].min(), df["TIME_PERIOD"].max()

### Drop non EU27_2020 countries

In [ ]:
eu27_2020 = ['BE', 'BG', 'CZ', 'DK', 'DE', 'EE', 'IE', 'EL', 'ES', 'FR', 'HR', 'IT', 'CY', 'LV', 'LT', 'LU', 'HU', 'MT', 'NL', 'AT', 'PL', 'PT', 'RO', 'SI', 'SK', 'FI', 'SE']

# Check wether the df contains all the EU27_2020 countries
df_countries = df["geo"].unique().tolist()
all_countries_in_df = all(country in df_countries for country in eu27_2020)

print(all_countries_in_df) # True, all EU27_2020 countries are present in the dataset

In [ ]:
# Filter the dataset to include only EU27_2020 countries
df = df[df["geo"].isin(eu27_2020)]

print(df["geo"].nunique()) # 27, only EU27_2020 countries are present in the dataset

### Decrease granularity of the dataset by aggregating the "Motor energy" column

- *Original values*: Alternative energy, Bi-fuel, Biodiesel, Bioethanol, Diesel, Diesel (excluding hybrids), Electricity, Hybrid diesel-electric, Plug-in hybrid diesel-electric, Hybrid electric-petrol, Plug-in hybrid petrol-electric, Natural gas, Hydrogen and fuel cells, Liquefied petroleum gases (LPG), Other, Petrol, Petrol (excluding hybrids), Total
- *Aggregated values*: Diesel, Diesel (excluding hybrids), Diesel hybrid, Diesel plug-in hybrid, Petrol, Petrol (excluding hybrids), Petrol hybrid, Petrol plug-in hybrid, Electricity, Total

In [ ]:
# Clean the "Motor energy" column by removing non-breaking spaces and stripping whitespace
df["Motor energy"] = (
    df["Motor energy"]
    .str.replace("\xa0", "", regex=False)
    .str.strip()
)

In [ ]:
motor_energy_map = {
    "Alternative energy": "Other",
    "Bi-fuel": "Other",
    "Biodiesel": "Other",
    "Bioethanol": "Other",
    "Natural gas": "Other",
    "Hydrogen and fuel cells": "Other",
    "Liquefied petroleum gases (LPG)": "Other",
    "Other": "Other",

    "Diesel": "Diesel",
    "Diesel (excluding hybrids)": "Diesel (excluding hybrids)",
    "Hybrid diesel-electric": "Diesel hybrid",
    "Plug-in hybrid diesel-electric": "Diesel plug-in hybrid",

    "Petrol": "Petrol",
    "Petrol (excluding hybrids)": "Petrol (excluding hybrids)",
    "Hybrid electric-petrol": "Petrol hybrid",
    "Plug-in hybrid petrol-electric": "Petrol plug-in hybrid",

    "Electricity": "Electricity",

    "Total": "Total"
}
df["Motor energy"] = df["Motor energy"].replace(motor_energy_map)

df["Motor energy"].unique()

### Analysis on the detailed and aggregated "Motor energy" categories (Diesel, Petrol)

For "TIME_PERIOD" <= 2021, the sum of the "OBS_VALUE" column for detailed categories (i.e. Diesel (excluding hybrids), Diesel hybrid, Diesel plug-in hybrid) does not match the aggregated categories (i.e. Diesel). 

This is because before 2021 memeber states were not required to report the detailed categories, so the aggregated categories were reported instead. Some member states reported the detailed categories, while others reported the aggregated categories.

In [ ]:
MOTOR_ENERGY = "Petrol" # Choose "Petrol" or "Diesel"

comparison_df = (
    df[df["Motor energy"].str.startswith(MOTOR_ENERGY)]
    .pivot_table(
        index="TIME_PERIOD",
        columns="Motor energy",
        values="OBS_VALUE",
        aggfunc="sum",
    )
)

# Check if the sum of detailed categories matches the aggregated category
comparison_df["Sum of " + MOTOR_ENERGY + " detailed categories"] = (
    comparison_df[MOTOR_ENERGY + " (excluding hybrids)"]
    + comparison_df[MOTOR_ENERGY + " hybrid"]
    + comparison_df[MOTOR_ENERGY + " plug-in hybrid"]
)
comparison_df["Difference"] = comparison_df[MOTOR_ENERGY] - comparison_df["Sum of " + MOTOR_ENERGY + " detailed categories"]

comparison_df[[MOTOR_ENERGY, "Sum of " + MOTOR_ENERGY + " detailed categories", "Difference"]]

In [ ]:
YEAR = 2023

tmp = df[df["TIME_PERIOD"] == YEAR]

tmp.groupby("Motor energy")["OBS_VALUE"].sum()

**How do we handle this?** Maybe:
- if there is a 0 in the detailed categories, then take the value from the aggregated category because the detailed are not correct
- if detailed categories are != 0, we can use detailed categories for better analysis

In [ ]:
df.to_csv('../data/cleaned/2c_eurostat_new_passenger_cars_by_type_of_motor_energy.csv', index=False)
df.info()